# Voice Disorder Recognizer
### GMM + convolutional DNN · academic review

Real VOICED recordings. **Development validation results; final test is sealed unless explicitly evaluated.**

This is an academic prototype, not a medical diagnostic tool. Run cells in order. No cell here trains models or opens the test set.

In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display, Audio, Image
from voice_review.experiment import predict
ROOT = Path.cwd()
assert (ROOT / "models/gmm.joblib").exists(), "Open this notebook from the project folder."
info = json.loads((ROOT / "results/development_metrics.json").read_text())
audit = json.loads((ROOT / "data/audit.json").read_text())
print(f"{audit['records']} records | {info['development_speakers']} development | {info['sealed_test_speakers']} reserved test")

## 1. Method
Audit identities and labels → freeze speaker partitions → central three-second crop → MFCC and voice-quality features → fold-local transforms → GMM / DNN → grouped validation.

GMM pools MFCCs; the DNN uses their time sequence. The same source recordings and feature families are compared. Metadata and diagnosis strings are never predictors.

In [ ]:
selection = json.loads((ROOT / "results/selection.json").read_text())
display(pd.DataFrame([{k:v for k,v in row.items() if k != "pooled_oof_metrics"} for row in selection.values()]))

## 2. Preliminary validation
These folds also selected settings; performance is not an unbiased final estimate. Balanced accuracy gives equal weight to healthy and pathological recall.

In [ ]:
display(pd.DataFrame(info["models"]).T.drop(columns=["confusion_matrix"]))
display(Image(filename=str(ROOT / "results/figures/confusion_matrices.png")))
display(Image(filename=str(ROOT / "results/figures/roc_curves.png")))

## 3. Working audio prediction
Run this cell to activate the controls. Keep **Speaker-held-out demo** selected, choose an example and click **Analyze example**. The default voice003.wav is real audio from a speaker excluded from those fold models' fitting and early stopping.

These are **development validation** predictions: settings were selected using validation results, so this is not a completely independent final test. For your own new WAV/FLAC, choose **Full-development models**, then **Choose audio** and **Analyze chosen file**. Full-development models have seen the bundled examples. Scores are uncalibrated, not medical probabilities.

In [ ]:
from voice_review.demo import make_demo, show_recording
demo = make_demo(ROOT)
display(demo)

## 4. Optional local-path alternative
If the controls are unavailable, set `audio_path` below to a local WAV/FLAC containing at least three seconds of sustained /a/. Displayed audio and results can be saved in notebook outputs; clear personal recordings before sharing.

In [ ]:
audio_path = None  # Example: ROOT / "examples" / "voice001.wav"
if audio_path is not None:
    try:
        result = show_recording(audio_path, ROOT)
    except ValueError as exc:
        print("Recording not accepted:", exc)
else:
    print("Use the controls above, or set audio_path for the local-path alternative.")

## 5. Explainability and limitations
Permutation analysis measures prediction sensitivity to feature groups, not causal disease mechanisms. Small subgroups do not support fairness claims. One controlled dataset cannot demonstrate generalization to real-world clinical recordings.

In [ ]:
display(Image(filename=str(ROOT / "results/figures/permutation_importance.png")))
print(audit["identity_basis"])
print("Final test:", "evaluated" if (ROOT / "results/final_metrics.json").exists() else "sealed")

## 6. Training stability
Four fixed seed runs on the original development folds. Initialization, minibatch order and the internal stopping subset vary; the architecture and outer folds stay fixed. Every run is shown, and the original saved models are retained. This does not measure uncertainty across new datasets.

In [ ]:
stability_path = ROOT / "results/robustness/summary.json"
if stability_path.exists():
    stability = json.loads(stability_path.read_text())
    display(pd.DataFrame(stability["summary"]).T)
    display(Image(filename=str(ROOT / "results/figures/seed_stability.png")))
else:
    print("Optional development seed analysis has not been run.")

## 7. References
Gupta R, Jin CT, Gunjawate DR, Nguyen DD, Stasak B, Chacon AM, Madill C. (2026).
Voice disorders classification using machine learning: a scoping review. Frontiers in Digital Health 8:1800132.
https://doi.org/10.3389/fdgth.2026.1800132

Cesari U, De Pietro G, Marciano E, Niri C, Sannino G, Verde L. (2018).
A new database of healthy and pathological voices. Computers & Electrical Engineering 68:310–321.
https://doi.org/10.1016/j.compeleceng.2018.04.008

VOICED Database, version 1.0.0. https://doi.org/10.13026/C25Q2N
https://physionet.org/content/voiced/1.0.0/ — Open Data Commons Attribution License v1.0.
License: https://opendatacommons.org/licenses/by/1-0/

Pollard TJ et al. (2026). PhysioNet as a global platform for biomedical research. Nature Health.
https://doi.org/10.1038/s44360-026-00096-z
